# Green NLP Benchmark Lab — Notebook 1: Baseline Measurement — coding workshop (advanced)

**Session 1 of 3.** Today you build the *baseline* version of a small pipeline that answers
questions about a handful of documents. You will measure how much energy and estimated CO2
it costs to run, and how good the answers are. Sessions 2 and 3 will change **one thing** each
time (quantization, then retrieval) and you'll compare against what you measure today.

Part 1 below is that QA baseline. Parts 2 and 3 are a self-contained detour into machine translation, comparing four ways to translate a sentence, and three ways to evaluate the result, all by cost.

**Before you start:** `Runtime` → `Change runtime type` → select **T4 GPU**, then `Save`.

**At the end of this notebook** you will save a file called `leaderboard.csv`. Download it
from the Files sidebar (folder icon, left side) and keep it somewhere you can find it.

Expected time: 60–75 minutes for Part 1 (the QA baseline), plus another 40–60 minutes for Parts 2 and 3 (translation and evaluation) if your session has time for them — your instructor will tell you how much of this to do today.

## How this workshop works

This is the coding version of the notebook, **advanced level**. In some code cells, lines
are replaced by `# TODO` comments and `___`. Each exercise above such a cell says what the
code must do; the comments in the cell give the details. This version takes longer than
the expected time given above.

1. Replace every `___` with your code, then run the cell. Open the hints if you need them.
2. Run the check cell right below it. It prints `✓ Exercise N passed`, or an error
   message that says what to look at.
3. If you are stuck, the solution is below the check cell. Open it after you have tried.

A cell that still contains `___` stops with `NameError: name '___' is not defined`.
Later cells use your code, so finish each exercise before moving on.

Exercises marked **(advanced)** are only in this version. In them you implement an
equation from the lesson or a paper, or a technique that makes the pipeline cheaper to
run, and check it against the library or the baseline.

## Step 0 — Install and import

This installs the libraries we need: `transformers` (to load and run the model),
`accelerate` (to place the model on the GPU efficiently), and `codecarbon` (to measure
energy use and estimate CO2 emissions for the code that runs while it's active).

In [ ]:
!pip install -q transformers accelerate codecarbon pandas

In [ ]:
import time
import textwrap
import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from codecarbon import EmissionsTracker

print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU detected — the notebook will still run, just slower. "
          "Double check Runtime > Change runtime type > T4 GPU.")

## Step 1 — The corpus

This is our little "knowledge base": eight short documents about Green NLP itself
(fittingly). Our pipeline's job is to answer questions using *only* this text — no
outside knowledge allowed, even if the model happens to already know the answer.

Each document below is original text written for this course, summarizing findings from
real papers, cited at the end of each document.

In [ ]:
CORPUS = {
    "training_vs_inference": """
Training a large language model is a one-time, very expensive event: the organization
that builds the model spends a large amount of compute once. Inference — actually using
the trained model to answer a query — is comparatively cheap per request, but it happens
every single time anyone uses the model. Once a model is deployed and used by many people,
the accumulated cost of inference tends to dominate the model's total lifetime footprint,
even though any single inference looks negligible next to the training run.
Source: Patterson et al., "Carbon Emissions and Large Neural Network Training," 2021
(arXiv:2104.10350). Training GPT-3 was estimated to emit 552 metric tons of CO2-equivalent,
using about 1,287 megawatt-hours of electricity.
""",
    "data_centers": """
Data centers are the physical facilities that house the servers running most of the
world's AI models. According to the International Energy Agency, global data center
electricity consumption was about 415 terawatt-hours in 2024, roughly 1.5 percent of all
electricity consumed worldwide that year. That demand is growing fast: electricity use by
data centers rose by about 17 percent in 2025 alone, driven mostly by AI workloads, and is
projected to approach 950 terawatt-hours by 2030, or about 3 percent of global electricity
demand.
Source: International Energy Agency, "Energy and AI" and related data centre electricity
reports, 2024-2025.
""",
    "quantization_basics": """
Neural network weights are normally stored as 32-bit floating point numbers (FP32).
Quantization reduces the number of bits used to represent each weight — for example to
16-bit (FP16/BF16), 8-bit integers (INT8), or even lower. Fewer bits per number means a
smaller model file and less memory traffic when running it, at the cost of some numerical
precision. A useful analogy: quantizing a model is like reducing the number of colors used
to represent an image — the picture is still recognizable, but some fine detail is lost.
Whether this actually saves energy in practice depends on the hardware: it helps most when
the bottleneck is moving data in and out of memory, not raw computation.
Source: Grootendorst, "A Visual Guide to Quantization," 2024.
""",
    "measuring_carbon": """
Estimating the carbon footprint of running a model requires several assumptions: how much
electricity the hardware draws, the Power Usage Effectiveness (PUE) of the data center
(extra power spent on cooling and overhead), and the carbon intensity of the local
electricity grid — how much CO2 is emitted per unit of electricity, which varies a lot by
region and by time of day. Because of all these assumptions, published carbon estimates can
vary widely, and some influential early estimates have later been revised — sometimes by an
order of magnitude — once other researchers re-examined the methodology. This is one reason
this course emphasizes measuring your own numbers rather than only citing others' headline
figures.
""",
    "inference_cost_task_type": """
Not all models cost the same to run for the same task. A 2024 study compared task-specific
models (built and trained for one narrow job) against multi-purpose generative models (like
general-purpose chatbots) performing the exact same tasks. For extractive question
answering, the most efficient task-specific models emitted around 0.3 grams of
CO2-equivalent per 1,000 inferences, while general-purpose generative models emitted around
10 grams of CO2-equivalent per 1,000 inferences for the same task — roughly 33 times more.
The gap was consistent across several other tasks too, even when controlling for the number
of parameters.
Source: Luccioni, Jernite & Strubell, "Power Hungry Processing: Watts Driving the Cost of AI
Deployment?", FAccT 2024 (arXiv:2311.16863).
""",
    "rag_concept": """
Retrieval-Augmented Generation, or RAG, is a technique where a model is given relevant
external text — retrieved from a document collection — as part of its input at query time,
instead of relying only on what it "learned" during training. This lets a smaller, cheaper
model answer questions about information it was never trained on, without the cost of
retraining or fine-tuning a much larger model on that information. The retrieval step
usually uses a separate, lightweight embedding model to find which documents are relevant
before the language model ever sees them.
""",
    "small_models_trend": """
Between 2024 and 2026, there has been a strong trend toward small, efficient open-weight
language models (often called SLMs, small language models) in the 1-4 billion parameter
range, designed to run on a single consumer GPU or even a laptop CPU after quantization.
These models are trained to be very capable for their size, and combined with quantization
and retrieval, they let many real applications avoid depending on a large, general-purpose
model at all — often at a fraction of the inference cost, if the task doesn't need the
breadth of a much bigger model.
""",
    "transformer_basics": """
Modern language models are built from transformer layers. Inside each layer,
"self-attention" lets every position in the input sequence gather information from every
other position, which is what allows the model to relate words like "it" back to the noun
it refers to several words earlier. This mechanism is powerful but has a computational cost
that grows faster than the length of the input (roughly quadratically with sequence
length), and more parameters mean more numbers to multiply and move through memory for
every single token generated. This is the underlying reason why both model size and input
length are direct levers on energy cost.
""",
}

for name, text in CORPUS.items():
    print(f"--- {name} ({len(text.split())} words) ---")
print(f"\nTotal documents: {len(CORPUS)}")

## Step 2 — The quiz

These are the questions your pipeline needs to answer correctly, one per document, plus
the short reference answer we'll use to automatically check quality. This is a simple
substring/keyword check — good enough to compare configurations against each other in
this course.

In [ ]:
QA_PAIRS = [
    {
        "question": "According to Patterson et al. (2021), how many metric tons of CO2-equivalent did training GPT-3 emit?",
        "reference": ["552"],
    },
    {
        "question": "Per the IEA, roughly what percentage of global electricity consumption did data centers represent in 2024?",
        "reference": ["1.5"],
    },
    {
        "question": "What does the acronym PUE stand for?",
        "reference": ["power usage effectiveness"],
    },
    {
        "question": "What does carbon footprint estimation depend on besides hardware power draw and grid carbon intensity?",
        "reference": ["pue", "power usage effectiveness"],
    },
    {
        "question": "In the 2024 study on inference costs, about how many grams of CO2-equivalent do the most efficient task-specific models emit per 1,000 extractive question-answering inferences?",
        "reference": ["0.3"],
    },
    {
        "question": "What is the name of the technique where a model is given retrieved external text at query time instead of relying only on what it learned during training?",
        "reference": ["retrieval-augmented generation", "rag"],
    },
    {
        "question": "What size range (in billions of parameters) do small language models (SLMs) typically fall into?",
        "reference": ["1-4", "1 to 4", "1-4 billion"],
    },
    {
        "question": "What mechanism inside a transformer layer lets every position gather information from every other position?",
        "reference": ["self-attention", "attention"],
    },
]

print(f"Number of questions: {len(QA_PAIRS)}")

## Step 3 — Load the baseline model

We'll use **Qwen2.5-1.5B-Instruct** (Apache 2.0 license, no gated access needed). It's small
enough to load quickly and run comfortably on a free-tier Colab GPU.

The first time you run this cell, it downloads about 3GB of model weights, which can take a
few minutes depending on Colab's connection that day — this is normal, just let it finish.

*Optional stretch goal:* once everything works, you can change `MODEL_NAME`
below to a bigger model (e.g. `Qwen/Qwen2.5-3B-Instruct`) and re-run, to see how the
baseline numbers change with size alone.

In [ ]:
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto",
)
print(f"Loaded {MODEL_NAME}")
print(f"Parameter count: {sum(p.numel() for p in model.parameters()) / 1e9:.2f}B")

## Step 4 — The pipeline: stuff everything into context

This pipeline is deliberately the *naive* baseline: for every question, we hand the
model **all eight documents** plus the question, and let it find the answer itself. No
retrieval, no filtering — simple, and exactly the kind of "just throw the whole context at
a big model" approach that targeted retrieval is built to improve on.

### ✏️ Exercise 1 — Build the prompt and keep only the answer

Two gaps:

1. In `build_prompt`, join all documents into one `context` string: each document as
   `[name]`, a newline, and its text, with a blank line between documents.
2. In `answer_question`, `model.generate` returns the prompt tokens **followed by** the new
   tokens. Keep only the new ones.

<details><summary><b>Hints</b> (click to open)</summary>

- `text.strip()` removes the blank lines around each document.
- `inputs["input_ids"].shape[1]` is the number of tokens in the prompt.

</details>

In [ ]:
def build_prompt(question, corpus):
    context = ___  # TODO: every document as "[name]\ntext", separated by blank lines
    messages = [
        {
            "role": "system",
            "content": (
                "You are a concise assistant. Answer the question using ONLY the "
                "documents provided below. Answer in one short sentence."
            ),
        },
        {
            "role": "user",
            "content": f"Documents:\n{context}\n\nQuestion: {question}",
        },
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


def answer_question(question, corpus, max_new_tokens=60):
    prompt = build_prompt(question, corpus)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
        )
    generated = ___  # TODO: only the newly generated token ids
    return tokenizer.decode(generated, skip_special_tokens=True).strip()


sample_answer = answer_question(QA_PAIRS[0]["question"], CORPUS)
print("Q:", QA_PAIRS[0]["question"])
print("A:", sample_answer)

In [ ]:
# Check for Exercise 1. Run this cell as is.
_p = build_prompt("TEST QUESTION", CORPUS)
assert all(f"[{name}]" in _p for name in CORPUS), "every document name should appear in the prompt"
assert CORPUS["rag_concept"].strip() in _p, "document text should appear in the prompt"
assert "TEST QUESTION" in _p
assert isinstance(sample_answer, str) and sample_answer, "empty answer"
assert "Documents:" not in sample_answer, "the answer still contains the prompt: slice off the prompt tokens"
print("✓ Exercise 1 passed")

<details><summary><b>Solution to Exercise 1</b> (open it after you have tried)</summary>

```python
def build_prompt(question, corpus):
    context = "\n\n".join(f"[{name}]\n{text.strip()}" for name, text in corpus.items())
    messages = [
        {
            "role": "system",
            "content": (
                "You are a concise assistant. Answer the question using ONLY the "
                "documents provided below. Answer in one short sentence."
            ),
        },
        {
            "role": "user",
            "content": f"Documents:\n{context}\n\nQuestion: {question}",
        },
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


def answer_question(question, corpus, max_new_tokens=60):
    prompt = build_prompt(question, corpus)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
        )
    generated = output[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()


sample_answer = answer_question(QA_PAIRS[0]["question"], CORPUS)
print("Q:", QA_PAIRS[0]["question"])
print("A:", sample_answer)
```

</details>

## Step 5 — Measure: run the full quiz with CodeCarbon

Now we run all eight questions through the pipeline, with `EmissionsTracker` running the
whole time. It estimates energy use (kWh) and CO2-equivalent (kg) for the code that runs
inside the `tracker.start()` / `tracker.stop()` block.

**Known limitation, worth noticing:** CodeCarbon *estimates* emissions — it doesn't have a
wattmeter plugged into the wall. On some GPUs (including some Colab allocations) it can't
read power draw directly and falls back to a rougher estimate. If you see a warning about
an unrecognized GPU, that's this limitation in action, a live example of
the measurement-uncertainty point above.

You'll likely see two more CodeCarbon warnings later in this notebook, for the same
reason — both harmless:

- `Multiple instances of codecarbon are allowed to run at the same time` — this notebook
  creates a fresh tracker for almost every measurement; CodeCarbon is just noting that.
- `Unable to access geographical location ... falls back to using the backup API` —
  CodeCarbon tries to look up your local electricity grid's carbon intensity from your IP;
  if that lookup fails (campus network, VPN, the geolocation service being down) it falls
  back to a default intensity value instead. The *relative* comparisons this notebook
  makes (which approach is cheaper than which) stay valid either way, since the same
  fallback applies consistently to every measurement — but the *absolute* g CO2eq numbers
  are then less geographically precise — another live example of measurement uncertainty.

### ✏️ Exercise 2 — Measure the run

Start the CodeCarbon tracker and a clock before the loop. After the loop, store the
elapsed seconds in `elapsed` and the tracker's result in `emissions_kg`.

<details><summary><b>Hints</b> (click to open)</summary>

- `tracker.start()` starts measuring; `tracker.stop()` stops and returns kg CO2eq.
- `time.time()` is the current time in seconds.

</details>

In [ ]:
tracker = EmissionsTracker(
    project_name="green-nlp-baseline",
    measure_power_secs=1,
    save_to_file=False,
    log_level="warning",
)

results = []
# TODO: start the tracker and the clock
___

for pair in QA_PAIRS:
    answer = answer_question(pair["question"], CORPUS)
    results.append({"question": pair["question"], "reference": pair["reference"], "answer": answer})

# TODO: stop measuring; store elapsed (seconds) and emissions_kg
___

print(f"Ran {len(QA_PAIRS)} questions in {elapsed:.1f} seconds")
print(f"Estimated emissions: {emissions_kg * 1000:.4f} g CO2eq")

In [ ]:
# Check for Exercise 2. Run this cell as is.
assert len(results) == len(QA_PAIRS)
assert 0 < elapsed < 3600, elapsed
assert emissions_kg is not None and emissions_kg > 0, emissions_kg
print("✓ Exercise 2 passed")

<details><summary><b>Solution to Exercise 2</b> (open it after you have tried)</summary>

```python
tracker = EmissionsTracker(
    project_name="green-nlp-baseline",
    measure_power_secs=1,
    save_to_file=False,
    log_level="warning",
)

results = []
tracker.start()
start_time = time.time()

for pair in QA_PAIRS:
    answer = answer_question(pair["question"], CORPUS)
    results.append({"question": pair["question"], "reference": pair["reference"], "answer": answer})

elapsed = time.time() - start_time
emissions_kg = tracker.stop()

print(f"Ran {len(QA_PAIRS)} questions in {elapsed:.1f} seconds")
print(f"Estimated emissions: {emissions_kg * 1000:.4f} g CO2eq")
```

</details>

## Step 6 — Score quality

A simple, transparent scoring rule: an answer counts as correct if it contains (case-insensitive)
at least one of the reference keywords/phrases for that question — a quick proxy, good
enough to compare configurations against each other consistently.

### ✏️ Exercise 3 — Score the answers

Write `score_answer`: an answer is correct if it contains at least one of the
reference keywords, ignoring upper and lower case. Then compute `accuracy`, the fraction
of correct answers.

<details><summary><b>Hints</b> (click to open)</summary>

- Lowercase both sides before comparing.
- In Python, `True` counts as 1 in `sum`.

</details>

In [ ]:
def score_answer(answer, reference_list):
    # TODO: True if any reference keyword occurs in the answer, ignoring case
    ___


for r in results:
    r["correct"] = score_answer(r["answer"], r["reference"])

accuracy = ___  # TODO: fraction of results that are correct

for r in results:
    mark = "✓" if r["correct"] else "✗"
    print(f"{mark} Q: {r['question']}")
    print(f"   A: {r['answer']}\n")

print(f"Accuracy: {accuracy:.0%} ({sum(r['correct'] for r in results)}/{len(results)})")

In [ ]:
# Check for Exercise 3. Run this cell as is.
assert score_answer("It emitted 552 tonnes.", ["552"]) is True
assert score_answer("Power Usage Effectiveness", ["power usage effectiveness"]) is True
assert score_answer("I do not know.", ["552"]) is False
assert abs(accuracy - sum(score_answer(r["answer"], r["reference"]) for r in results) / len(results)) < 1e-9
print("✓ Exercise 3 passed")

<details><summary><b>Solution to Exercise 3</b> (open it after you have tried)</summary>

```python
def score_answer(answer, reference_list):
    answer_lower = answer.lower()
    return any(ref.lower() in answer_lower for ref in reference_list)


for r in results:
    r["correct"] = score_answer(r["answer"], r["reference"])

accuracy = sum(r["correct"] for r in results) / len(results)

for r in results:
    mark = "✓" if r["correct"] else "✗"
    print(f"{mark} Q: {r['question']}")
    print(f"   A: {r['answer']}\n")

print(f"Accuracy: {accuracy:.0%} ({sum(r['correct'] for r in results)}/{len(results)})")
```

</details>

### Step 6, continued — the same quiz with no documents

Every prompt above contains all eight documents, and the documents contain the answers. So
the quiz measures whether the model can *find* a fact it has been handed. How much does the
model know without them? Here is the same quiz, same model and same one-sentence
instruction, with the question alone. This is called **closed-book** question answering:
the model can only draw on what it learned in training.

### ✏️ Exercise 4 — The closed-book prompt

Complete the user message for `answer_closed_book`: the question alone, with no
documents.

<details><summary><b>Hints</b> (click to open)</summary>

- Compare with the user message in `build_prompt` (Exercise 1).
- An f-string: `f"Question: {question}"`.

</details>

In [ ]:
def answer_closed_book(question, max_new_tokens=60):
    messages = [
        {"role": "system",
         "content": "You are a concise assistant. Answer the question in one short sentence."},
        ___,  # TODO: the user message: "Question: " followed by the question
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    generated = output[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()


tracker_closed = EmissionsTracker(
    project_name="green-nlp-closed-book",
    measure_power_secs=1,
    save_to_file=False,
    log_level="warning",
)

results_closed = []
tracker_closed.start()
start_time = time.time()

for pair in QA_PAIRS:
    answer = answer_closed_book(pair["question"])
    results_closed.append({"question": pair["question"], "answer": answer,
                           "correct": score_answer(answer, pair["reference"])})

elapsed_closed = time.time() - start_time
emissions_closed_kg = tracker_closed.stop()
accuracy_closed = sum(r["correct"] for r in results_closed) / len(results_closed)

for r_docs, r_closed in zip(results, results_closed):
    print(f"Q: {r_docs['question']}")
    print(f"   with documents {'✓' if r_docs['correct'] else '✗'}  {r_docs['answer']}")
    print(f"   no documents   {'✓' if r_closed['correct'] else '✗'}  {r_closed['answer']}\n")

tokens_with_docs = len(tokenizer(build_prompt(QA_PAIRS[0]["question"], CORPUS))["input_ids"])
print(f"Accuracy with documents: {accuracy:.0%}   no documents: {accuracy_closed:.0%}")
print(f"Emissions with documents: {emissions_kg * 1000:.4f} g CO2eq   "
      f"no documents: {emissions_closed_kg * 1000:.4f} g CO2eq")
print(f"Prompt length for the first question: {tokens_with_docs} tokens with documents")

In [ ]:
# Check for Exercise 4. Run this cell as is.
assert len(results_closed) == len(QA_PAIRS)
assert all(isinstance(r["answer"], str) and r["answer"] for r in results_closed)
assert "Documents:" not in results_closed[0]["answer"]
print("✓ Exercise 4 passed")

<details><summary><b>Solution to Exercise 4</b> (open it after you have tried)</summary>

```python
def answer_closed_book(question, max_new_tokens=60):
    messages = [
        {"role": "system",
         "content": "You are a concise assistant. Answer the question in one short sentence."},
        {"role": "user", "content": f"Question: {question}"},
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    generated = output[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()


tracker_closed = EmissionsTracker(
    project_name="green-nlp-closed-book",
    measure_power_secs=1,
    save_to_file=False,
    log_level="warning",
)

results_closed = []
tracker_closed.start()
start_time = time.time()

for pair in QA_PAIRS:
    answer = answer_closed_book(pair["question"])
    results_closed.append({"question": pair["question"], "answer": answer,
                           "correct": score_answer(answer, pair["reference"])})

elapsed_closed = time.time() - start_time
emissions_closed_kg = tracker_closed.stop()
accuracy_closed = sum(r["correct"] for r in results_closed) / len(results_closed)

for r_docs, r_closed in zip(results, results_closed):
    print(f"Q: {r_docs['question']}")
    print(f"   with documents {'✓' if r_docs['correct'] else '✗'}  {r_docs['answer']}")
    print(f"   no documents   {'✓' if r_closed['correct'] else '✗'}  {r_closed['answer']}\n")

tokens_with_docs = len(tokenizer(build_prompt(QA_PAIRS[0]["question"], CORPUS))["input_ids"])
print(f"Accuracy with documents: {accuracy:.0%}   no documents: {accuracy_closed:.0%}")
print(f"Emissions with documents: {emissions_kg * 1000:.4f} g CO2eq   "
      f"no documents: {emissions_closed_kg * 1000:.4f} g CO2eq")
print(f"Prompt length for the first question: {tokens_with_docs} tokens with documents")
```

</details>

### ✏️ Exercise 5 — Estimate the compute from an equation (advanced)

A forward pass through a transformer costs about **2N floating-point operations
(FLOPs) per token**, where N is the number of parameters: each weight is used in one
multiplication and one addition (Kaplan et al. 2020, *Scaling Laws for Neural Language
Models*, arXiv:2001.08361). The model processes every prompt token once, and then every
generated token once, so for one question:

FLOPs ≈ 2 · N · (prompt tokens + generated tokens)

Complete the three functions, then run the cell. It estimates the compute of the run with
documents (`results`) and of the closed-book run (`results_closed`), and prints the ratio of
the two next to the ratios you measured. Count the generated tokens by tokenizing the
stored answer; this is close to, but not always exactly, what the model generated.

**Reflection:** do the estimated and the measured ratios agree? The equation counts
arithmetic only. What else takes time when a model generates text one token at a time?

<details><summary><b>Hints</b> (click to open)</summary>

- `tokenizer(text)["input_ids"]` is the list of token ids for `text`.
- For each result: `count_tokens(make_prompt(r["question"])) + count_tokens(r["answer"])`.
- `total += forward_flops(n_tokens)`.

</details>

In [ ]:
N_PARAMS = sum(p.numel() for p in model.parameters())


def closed_book_prompt(question):
    messages = [
        {"role": "system",
         "content": "You are a concise assistant. Answer the question in one short sentence."},
        {"role": "user", "content": f"Question: {question}"},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


def count_tokens(text):
    # TODO: the number of tokens in text, using the model's tokenizer
    ___


def forward_flops(n_tokens, n_params=N_PARAMS):
    # TODO: the 2N-per-token estimate for n_tokens tokens
    ___


def run_flops(results_list, make_prompt):
    """Estimated FLOPs of a whole run. make_prompt(question) returns the prompt text."""
    total = 0
    for r in results_list:
        # TODO: tokens processed for this question (prompt + answer); add their FLOPs to total
        ___
    return total

flops_docs = run_flops(results, lambda q: build_prompt(q, CORPUS))
flops_closed = run_flops(results_closed, closed_book_prompt)

print(f"With documents: {flops_docs:.2e} FLOPs")
print(f"Closed-book:    {flops_closed:.2e} FLOPs")
print(f"Ratio, estimated FLOPs:  {flops_docs / flops_closed:5.1f}x")
print(f"Ratio, measured time:    {elapsed / elapsed_closed:5.1f}x")
print(f"Ratio, measured CO2eq:   {emissions_kg / emissions_closed_kg:5.1f}x")

In [ ]:
# Check for Exercise 5. Run this cell as is.
_q = QA_PAIRS[0]["question"]
assert count_tokens(build_prompt(_q, CORPUS)) == len(tokenizer(build_prompt(_q, CORPUS))["input_ids"])
assert forward_flops(1000, n_params=1e9) == 2e12, "1000 tokens through 1B parameters: 2e12 FLOPs"
_expected = sum(2 * N_PARAMS * (count_tokens(build_prompt(r["question"], CORPUS)) + count_tokens(r["answer"]))
                for r in results)
assert abs(flops_docs - _expected) / _expected < 1e-9, "count the prompt tokens and the answer tokens"
assert flops_docs > flops_closed, "the prompts with documents are much longer"
print("✓ Exercise 5 passed")

<details><summary><b>Solution to Exercise 5</b> (open it after you have tried)</summary>

```python
N_PARAMS = sum(p.numel() for p in model.parameters())


def closed_book_prompt(question):
    messages = [
        {"role": "system",
         "content": "You are a concise assistant. Answer the question in one short sentence."},
        {"role": "user", "content": f"Question: {question}"},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


def count_tokens(text):
    return len(tokenizer(text)["input_ids"])


def forward_flops(n_tokens, n_params=N_PARAMS):
    return 2 * n_params * n_tokens


def run_flops(results_list, make_prompt):
    """Estimated FLOPs of a whole run. make_prompt(question) returns the prompt text."""
    total = 0
    for r in results_list:
        n_tokens = count_tokens(make_prompt(r["question"])) + count_tokens(r["answer"])
        total += forward_flops(n_tokens)
    return total

flops_docs = run_flops(results, lambda q: build_prompt(q, CORPUS))
flops_closed = run_flops(results_closed, closed_book_prompt)

print(f"With documents: {flops_docs:.2e} FLOPs")
print(f"Closed-book:    {flops_closed:.2e} FLOPs")
print(f"Ratio, estimated FLOPs:  {flops_docs / flops_closed:5.1f}x")
print(f"Ratio, measured time:    {elapsed / elapsed_closed:5.1f}x")
print(f"Ratio, measured CO2eq:   {emissions_kg / emissions_closed_kg:5.1f}x")
```

</details>

### ✏️ Exercise 6 — Read the documents once: prefix caching (advanced)

Every prompt in Step 5 starts with the same system message and the same eight
documents. Only the question at the end changes. Still, the model processes those shared
tokens again for each of the eight questions.

While a model reads a prompt, each attention layer computes a key vector and a value
vector for every token. They are kept in the **KV cache**, so that later tokens can attend
to them without computing them again. A cache built once for the shared start of the
prompt (the **prefix**) can be reused for every question. This is called **prefix
caching**.

1. `prefix_text` is the part of the prompt before `"Question: "`.
2. Inside the measured block, run the model once on the prefix and keep the cache.
3. In `answer_with_prefix_cache`, pass a **copy** of that cache to `model.generate`.
   `generate` then processes only the tokens that are not in the cache yet. It also adds
   the new tokens to the cache it is given, so each question needs its own copy.

The cell measures the new run like Step 5. The answers should match the baseline's. In
float16, the order of the arithmetic changes the last digits of some values, so an
answer or two may differ by a word.

**Reflection:** compare the saving in time with the share of FLOPs that the documents
account for in the previous exercise. Which part of answering a question does prefix
caching remove, and which part does it leave?

<details><summary><b>Hints</b> (click to open)</summary>

- `full_example.index("Question: ")` is the position where the question starts; slice up to it.
- `inputs["input_ids"][0, :prefix_ids.shape[1]]` are the prompt's first tokens; compare them with `torch.equal`.
- Filling the cache: `model(prefix_ids, past_key_values=prefix_cache, use_cache=True)`.
- `copy.deepcopy(prefix_cache)` makes a copy that `generate` can change.

</details>

In [ ]:
import copy
from transformers import DynamicCache

full_example = build_prompt("PLACEHOLDER", CORPUS)
prefix_text = ___  # TODO: the part of full_example before "Question: "
prefix_ids = tokenizer(prefix_text, return_tensors="pt")["input_ids"].to(model.device)


def answer_with_prefix_cache(question, max_new_tokens=60):
    inputs = tokenizer(build_prompt(question, CORPUS), return_tensors="pt").to(model.device)
    # TODO: assert that the prompt's first tokens are exactly prefix_ids
    ___
    with torch.no_grad():
        output = model.generate(
            **inputs,
            past_key_values=___,  # TODO: a copy of the prefix cache
            max_new_tokens=max_new_tokens,
            do_sample=False,
        )
    generated = output[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()


tracker_cached = EmissionsTracker(
    project_name="green-nlp-prefix-cache",
    measure_power_secs=1,
    save_to_file=False,
    log_level="warning",
)

results_cached = []
tracker_cached.start()
start_time = time.time()

prefix_cache = DynamicCache()
with torch.no_grad():
    # TODO: one forward pass over prefix_ids that fills prefix_cache
    # (pass past_key_values=prefix_cache and use_cache=True)
    ___

for pair in QA_PAIRS:
    results_cached.append({"question": pair["question"],
                           "answer": answer_with_prefix_cache(pair["question"])})

elapsed_cached = time.time() - start_time
emissions_cached_kg = tracker_cached.stop()

same = sum(r["answer"] == c["answer"] for r, c in zip(results, results_cached))
print(f"Prefix: {prefix_ids.shape[1]} tokens, computed once instead of {len(QA_PAIRS)} times")
print(f"Same answer as the baseline: {same}/{len(QA_PAIRS)}")
print(f"Baseline:       {elapsed:6.1f} s  {emissions_kg * 1000:.4f} g CO2eq")
print(f"Prefix caching: {elapsed_cached:6.1f} s  {emissions_cached_kg * 1000:.4f} g CO2eq")

In [ ]:
# Check for Exercise 6. Run this cell as is.
assert full_example.startswith(prefix_text) and full_example[len(prefix_text):].startswith("Question: "), \
    "prefix_text should be everything before \"Question: \""
assert prefix_cache.get_seq_length() == prefix_ids.shape[1], \
    "the prefix cache grew: pass a copy (copy.deepcopy) to generate, and fill it only once"
assert len(results_cached) == len(QA_PAIRS)
assert same >= len(QA_PAIRS) - 2, f"only {same} answers match the baseline: is the cache built from the right prefix?"
print("✓ Exercise 6 passed")

<details><summary><b>Solution to Exercise 6</b> (open it after you have tried)</summary>

```python
import copy
from transformers import DynamicCache

full_example = build_prompt("PLACEHOLDER", CORPUS)
prefix_text = full_example[:full_example.index("Question: ")]
prefix_ids = tokenizer(prefix_text, return_tensors="pt")["input_ids"].to(model.device)


def answer_with_prefix_cache(question, max_new_tokens=60):
    inputs = tokenizer(build_prompt(question, CORPUS), return_tensors="pt").to(model.device)
    assert torch.equal(inputs["input_ids"][0, :prefix_ids.shape[1]], prefix_ids[0])
    with torch.no_grad():
        output = model.generate(
            **inputs,
            past_key_values=copy.deepcopy(prefix_cache),
            max_new_tokens=max_new_tokens,
            do_sample=False,
        )
    generated = output[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()


tracker_cached = EmissionsTracker(
    project_name="green-nlp-prefix-cache",
    measure_power_secs=1,
    save_to_file=False,
    log_level="warning",
)

results_cached = []
tracker_cached.start()
start_time = time.time()

prefix_cache = DynamicCache()
with torch.no_grad():
    model(prefix_ids, past_key_values=prefix_cache, use_cache=True)

for pair in QA_PAIRS:
    results_cached.append({"question": pair["question"],
                           "answer": answer_with_prefix_cache(pair["question"])})

elapsed_cached = time.time() - start_time
emissions_cached_kg = tracker_cached.stop()

same = sum(r["answer"] == c["answer"] for r, c in zip(results, results_cached))
print(f"Prefix: {prefix_ids.shape[1]} tokens, computed once instead of {len(QA_PAIRS)} times")
print(f"Same answer as the baseline: {same}/{len(QA_PAIRS)}")
print(f"Baseline:       {elapsed:6.1f} s  {emissions_kg * 1000:.4f} g CO2eq")
print(f"Prefix caching: {elapsed_cached:6.1f} s  {emissions_cached_kg * 1000:.4f} g CO2eq")
```

</details>

**Reflection:** Which questions did the model answer correctly with no documents, and
which did it miss? For the misses, did it say it didn't know, or did it give a specific
answer anyway? Compare the two accuracy numbers and the two emissions numbers: what do the
documents buy, and what do they cost?

## Step 7 — Log these runs to your leaderboard

Two rows go into your leaderboard. `baseline-fp16` is the reference every later session
compares against: future rows use the same questions and scoring, changing one thing at a
time — a quantized version of this exact pipeline, then a retrieval-augmented one.
`closed-book-fp16` records the no-documents run, as a reference point for what the
documents contribute.

### ✏️ Exercise 7 — Log the closed-book row

Fill in the measured values for the `closed-book-fp16` row.

<details><summary><b>Hints</b> (click to open)</summary>

- The closed-book run stored its results in variables ending in `_closed`.
- Look at the `baseline-fp16` row just above for the pattern.

</details>

In [ ]:
def tracker_energy_kwh(tracker_obj):
    try:
        return float(tracker_obj.final_emissions_data.energy_consumed)
    except (AttributeError, TypeError):
        return None


n_params_billion = round(sum(p.numel() for p in model.parameters()) / 1e9, 2)

leaderboard = pd.DataFrame([
    {
        "config": "baseline-fp16",
        "model": MODEL_NAME,
        "params_billion": n_params_billion,
        "n_questions": len(QA_PAIRS),
        "duration_sec": round(elapsed, 2),
        "energy_kwh": tracker_energy_kwh(tracker),
        "emissions_kg_co2eq": emissions_kg,
        "accuracy": accuracy,
    },
    {
        "config": "closed-book-fp16",
        "model": MODEL_NAME,
        "params_billion": n_params_billion,
        "n_questions": len(QA_PAIRS),
        "duration_sec": ___,  # TODO: closed-book duration, rounded to 2 decimals
        "energy_kwh": ___,  # TODO: closed-book energy, from its tracker
        "emissions_kg_co2eq": ___,
        "accuracy": ___,
    },
])
leaderboard.to_csv("leaderboard.csv", index=False)
leaderboard

In [ ]:
# Check for Exercise 7. Run this cell as is.
_row = leaderboard[leaderboard["config"] == "closed-book-fp16"].iloc[0]
assert _row["emissions_kg_co2eq"] == emissions_closed_kg
assert _row["accuracy"] == accuracy_closed
assert _row["duration_sec"] == round(elapsed_closed, 2)
print("✓ Exercise 7 passed")

<details><summary><b>Solution to Exercise 7</b> (open it after you have tried)</summary>

```python
def tracker_energy_kwh(tracker_obj):
    try:
        return float(tracker_obj.final_emissions_data.energy_consumed)
    except (AttributeError, TypeError):
        return None


n_params_billion = round(sum(p.numel() for p in model.parameters()) / 1e9, 2)

leaderboard = pd.DataFrame([
    {
        "config": "baseline-fp16",
        "model": MODEL_NAME,
        "params_billion": n_params_billion,
        "n_questions": len(QA_PAIRS),
        "duration_sec": round(elapsed, 2),
        "energy_kwh": tracker_energy_kwh(tracker),
        "emissions_kg_co2eq": emissions_kg,
        "accuracy": accuracy,
    },
    {
        "config": "closed-book-fp16",
        "model": MODEL_NAME,
        "params_billion": n_params_billion,
        "n_questions": len(QA_PAIRS),
        "duration_sec": round(elapsed_closed, 2),
        "energy_kwh": tracker_energy_kwh(tracker_closed),
        "emissions_kg_co2eq": emissions_closed_kg,
        "accuracy": accuracy_closed,
    },
])
leaderboard.to_csv("leaderboard.csv", index=False)
leaderboard
```

</details>

### Where this row fits

In [ ]:
import matplotlib.pyplot as plt

sessions = ["Session 1\n(baseline, today)", "Session 2\n(quantized)", "Session 3\n(+ retrieval)"]
today_value = emissions_kg * 1000
heights = [today_value, today_value, today_value]
colors = ["tab:blue", "lightgray", "lightgray"]
labels = [f"{today_value:.4f} g CO2eq\nmeasured today",
          "measured\nnext session",
          "measured\nin Session 3"]

fig, ax = plt.subplots(figsize=(6.5, 3.5))
bars = ax.bar(sessions, heights, color=colors)
bars[1].set_hatch("//")
bars[2].set_hatch("//")
for bar, label in zip(bars, labels):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() * 1.03, label,
            ha="center", va="bottom", fontsize=9)
ax.set_ylim(0, today_value * 1.4)
ax.get_yaxis().set_visible(False)
for spine in ("top", "right", "left"):
    ax.spines[spine].set_visible(False)
ax.set_title("baseline-fp16: the row later sessions compare against")
plt.tight_layout()
plt.show()

In [ ]:
import os

if os.path.exists("leaderboard.csv"):
    print("Saved leaderboard.csv in this Colab session.")
    print("To download it: click the folder icon in the left sidebar, find")
    print("leaderboard.csv, right-click it, and choose Download.")
else:
    print("leaderboard.csv not found — re-run the cell above first.")

## Part 2 — Machine translation: four eras, one sentence

The sentence:

> *"Ás veces é mellor estar calado. Unha tía miña, que todo o solucionaba con ditos e
> refráns, dicía algo así como 'un é escravo do que di e dono do que cala'."*

It is Galician — a low-resource language — from the novel
*Speechwriter* by Paulo Naseiro. In the lesson, the fragment "unha tía miña" was aligned by hand
with attention.

Now you are going to translate it with **four different kinds of system**, tracing that
same historical arc:

1. **Rule-based** (1970s-2000s style) — [Apertium](https://apertium.org), a real,
   actively maintained rule-based MT engine, called through its public API
2. **Statistical** (1990s style) — IBM Model 1, which learns word translations from
   example sentence pairs, then translates word by word
3. **Neural, dedicated** (2015-style architecture) — a small model trained for exactly one
   job: Galician → English
4. **General-purpose LLM** (today) — the same model you just used for question answering,
   simply asked to translate instead

Approach 2 is a real algorithm on a toy training set: 24 short phrase pairs, written for
this notebook. Approaches 1, 3, and 4 are real systems. All four translate the whole
passage.

*Want to use the first chapter of* Speechwriter*, or a different Galician text entirely, for
approaches 1, 3, and 4?* Ask your instructor for the chapter text, or paste in any Galician
text you like in Step 9 — or flip the direction and translate English **into** Galician
(see the stretch goal at the end of Part 2).

Expected additional time: 40–60 minutes for Part 2 and Part 3 together.

## Step 8 — Install the extra piece we need, and a shared helper

The general-purpose model is already loaded from Part 1. The dedicated translation model
uses a different tokenizer library (`sentencepiece`), so we install that here. We also
define one small helper function, reused for every measurement in Parts 2 and 3, that pulls
the total energy out of a `CodeCarbon` tracker the same way Step 7 did.

In [ ]:
!pip install -q sentencepiece sacremoses

from transformers import AutoModelForSeq2SeqLM


def get_tracker_energy_kwh(tracker):
    for attr_path in ("final_emissions_data.energy_consumed", "_total_energy.kWh"):
        obj = tracker
        try:
            for part in attr_path.split("."):
                obj = getattr(obj, part)
            return float(obj)
        except Exception:
            continue
    return None

## Step 9 — The text

`FULL_TEXT_GL` is the whole passage. All four translation systems below work on it
directly. Replace it with a longer passage or a different Galician text if you like — see
the stretch goal at the end of Part 2.

In [ ]:
FULL_TEXT_GL = (
    "Ás veces é mellor estar calado. Unha tía miña, que todo o solucionaba con ditos e "
    "refráns, dicía algo así como \"un é escravo do que di e dono do que cala\". Sempre me "
    "pareceu un sabio aforismo, e se cadra debería terlle feito caso nalgunhas ocasións, "
    "aquelas que, aínda que raras, contribuíron a dar bruscos cambios de sentido á miña vida."
)

print(f"Full text ({len(FULL_TEXT_GL.split())} words):\n{FULL_TEXT_GL}")

## Step 10 — Approach 1: rule-based, for real (1970s-2000s style)

A real system this time. [Apertium](https://apertium.org) is an open-source, actively
maintained rule-based MT engine: morphological analysis, part-of-speech disambiguation,
structural transfer rules, then generation — no statistics, no neural network anywhere in
the pipeline. It has a real Galician→English pair, and we call its public translation API
directly.

**Known limitation, worth noticing:** this translation happens on Apertium's own server.
The CodeCarbon numbers below measure *our local network wait*, standing in for a
server-side compute cost that stays invisible to us — genuinely hard to cost out
honestly, worth flagging rather than glossing over.

In [ ]:
import requests

APERTIUM_URL = "https://apertium.org/apy/translate"
APERTIUM_FALLBACK = (
    "A mine aunt, than everything solved it with said and sayings, said something as "
    "well as one is enslaved of the that says and owner of the that cove."
)


def translate_rule_based(text, langpair="glg|eng"):
    response = requests.get(APERTIUM_URL, params={"langpair": langpair, "q": text}, timeout=15)
    response.raise_for_status()
    return response.json()["responseData"]["translatedText"]


tracker_rule = EmissionsTracker(
    project_name="green-mt-rule-based",
    measure_power_secs=1,
    save_to_file=False,
    log_level="warning",
)
tracker_rule.start()
start_time = time.time()
try:
    translation_rule = translate_rule_based(FULL_TEXT_GL)
except Exception as e:
    print(f"Apertium API call failed ({e}) — using a real response captured earlier.")
    translation_rule = APERTIUM_FALLBACK
elapsed_rule = time.time() - start_time
emissions_rule_kg = tracker_rule.stop()

print(f"\"{FULL_TEXT_GL}\"")
print(f"-> \"{translation_rule}\"")
print(f"({elapsed_rule:.4f}s, {emissions_rule_kg * 1000:.6f} g CO2eq measured locally — the")
print("actual translation ran on Apertium's server.)")

### Is this Galician being under-served, or is Apertium just bad?

Neither, it turns out. Below is the exact same engine and the exact same
`translate_rule_based` function, translating two plain, non-idiomatic Spanish sentences —
Spanish is one of Apertium's most mature language pairs, nothing like Galician's smaller
coverage. If the garbled output above were really about Galician being under-resourced,
or Apertium just being a bad tool, plain Spanish sentences should look just as broken.

In [ ]:
SPANISH_PLAIN_SENTENCES = [
    "El gato negro duerme en la silla de la cocina todos los días.",
    "Mi hermana vive en una ciudad grande cerca del mar.",
]

SPANISH_FALLBACKS = [
    "The black cat sleeps in the chair of the cookery every day.",
    "My sister lives in a big city near of the mar.",
]

for sentence, fallback in zip(SPANISH_PLAIN_SENTENCES, SPANISH_FALLBACKS):
    try:
        translation = translate_rule_based(sentence, langpair="spa|eng")
    except Exception as e:
        print(f"Apertium API call failed ({e}) — using a real response captured earlier.")
        translation = fallback
    print(f"\"{sentence}\"")
    print(f"-> \"{translation}\"\n")

print("Correct word order, correct tense, mostly the right words — genuinely usable for")
print("plain declarative text. So the garbled Galician output above isn't Apertium being")
print("broken, and it isn't Galician being under-supported: it's rule-based MT hitting its")
print("real limitation — no representation of meaning, just structural substitution — on a")
print("sentence chosen specifically because it's proverb-heavy and idiomatic.")

**Reflection:** Look closely at the second sentence's output: *"My sister lives in a
big city near of the mar."* Something is genuinely wrong with it, in two separate ways —
one word problem and one grammar problem. What are they? (Try translating "mar" by
itself, and "cerca de" by itself, to investigate.)

## Step 11 — Approach 2: statistical, IBM Model 1 (1990s style)

A real implementation this time, not hand-invented: NLTK's `IBMModel1`, the actual
foundational word-alignment algorithm from the statistical MT research literature (Brown
et al., early 1990s). It is trained here on a small, hand-built Galician-English parallel
corpus --- nowhere near what a real system of that era would have used, but a real
algorithm learning real, if sparse, translation probabilities from real example
sentences, not a hand-written dictionary.

Decoding is deliberately the simplest possible version: pick the most likely English word
for each Galician word, in the original word order, with no language model and no
reordering search. That simplicity is the actual point, not a shortcut --- see the note
after the output below.

### ✏️ Exercise 8 — Pick the most probable translation of a word

In `best_translation`, `candidates` maps each English word to P(English word |
Galician word). Return the English word with the highest probability, or `None` if even
that probability is below `min_prob`.

<details><summary><b>Hints</b> (click to open)</summary>

- `candidates.items()` gives `(word, probability)` pairs; `kv[1]` is the probability.
- Return `None` when `prob` is smaller than `min_prob`.

</details>

In [ ]:
!pip install -q nltk
from nltk.translate import AlignedSent, IBMModel1
import re

BITEXT_PAIRS = [
    ("ás veces", "sometimes"),
    ("é mellor estar calado", "it is better to stay quiet"),
    ("unha tía miña", "an aunt of mine"),
    ("a miña tía", "my aunt"),
    ("todo", "everything"),
    ("solucionaba todo", "solved everything"),
    ("con ditos e refráns", "with sayings and proverbs"),
    ("dicía algo", "used to say something"),
    ("dicía algo así", "used to say something like this"),
    ("un é escravo", "one is a slave"),
    ("escravo do que di", "slave of what one says"),
    ("dono do que cala", "master of what one keeps quiet"),
    ("sempre me pareceu", "it always seemed to me"),
    ("un sabio aforismo", "a wise saying"),
    ("se cadra debería", "maybe i should have"),
    ("debería terlle feito caso", "should have paid heed to it"),
    ("nalgunhas ocasións", "on some occasions"),
    ("aquelas ocasións raras", "those rare occasions"),
    ("aínda que raras", "even though rare"),
    ("contribuíron a dar", "contributed to giving"),
    ("bruscos cambios", "sudden changes"),
    ("cambios de sentido", "changes of meaning"),
    ("á miña vida", "to my life"),
    ("a miña vida", "my life"),
]

tracker_stat = EmissionsTracker(
    project_name="green-mt-statistical",
    measure_power_secs=1,
    save_to_file=False,
    log_level="warning",
)
tracker_stat.start()
start_time = time.time()

bitext = [AlignedSent(en.split(), gl.split()) for gl, en in BITEXT_PAIRS]
ibm1_model = IBMModel1(bitext, 20)


def best_translation(src_word, table, min_prob=0.05):
    candidates = {tgt: table[tgt].get(src_word, 0.0) for tgt in table}
    if not candidates:
        return None
    # TODO: the most probable target word, or None below min_prob
    ___


def translate_statistical(text, table):
    words = re.findall(r"[a-záéíóúñ]+", text.lower())
    out = [best_translation(w, table) or f"*{w}" for w in words]
    return " ".join(out)


translation_stat = translate_statistical(FULL_TEXT_GL, ibm1_model.translation_table)
elapsed_stat = time.time() - start_time
emissions_stat_kg = tracker_stat.stop()

print(translation_stat)
print(f"\n({elapsed_stat:.4f}s, {emissions_stat_kg * 1000:.6f} g CO2eq)")

In [ ]:
# Check for Exercise 8. Run this cell as is.
_t = {"aunt": {"tía": 0.9}, "my": {"tía": 0.1, "miña": 0.8}}
assert best_translation("tía", _t) == "aunt"
assert best_translation("miña", _t) == "my"
assert best_translation("xyz", _t) is None
print("✓ Exercise 8 passed")

<details><summary><b>Solution to Exercise 8</b> (open it after you have tried)</summary>

```python
!pip install -q nltk
from nltk.translate import AlignedSent, IBMModel1
import re

BITEXT_PAIRS = [
    ("ás veces", "sometimes"),
    ("é mellor estar calado", "it is better to stay quiet"),
    ("unha tía miña", "an aunt of mine"),
    ("a miña tía", "my aunt"),
    ("todo", "everything"),
    ("solucionaba todo", "solved everything"),
    ("con ditos e refráns", "with sayings and proverbs"),
    ("dicía algo", "used to say something"),
    ("dicía algo así", "used to say something like this"),
    ("un é escravo", "one is a slave"),
    ("escravo do que di", "slave of what one says"),
    ("dono do que cala", "master of what one keeps quiet"),
    ("sempre me pareceu", "it always seemed to me"),
    ("un sabio aforismo", "a wise saying"),
    ("se cadra debería", "maybe i should have"),
    ("debería terlle feito caso", "should have paid heed to it"),
    ("nalgunhas ocasións", "on some occasions"),
    ("aquelas ocasións raras", "those rare occasions"),
    ("aínda que raras", "even though rare"),
    ("contribuíron a dar", "contributed to giving"),
    ("bruscos cambios", "sudden changes"),
    ("cambios de sentido", "changes of meaning"),
    ("á miña vida", "to my life"),
    ("a miña vida", "my life"),
]

tracker_stat = EmissionsTracker(
    project_name="green-mt-statistical",
    measure_power_secs=1,
    save_to_file=False,
    log_level="warning",
)
tracker_stat.start()
start_time = time.time()

bitext = [AlignedSent(en.split(), gl.split()) for gl, en in BITEXT_PAIRS]
ibm1_model = IBMModel1(bitext, 20)


def best_translation(src_word, table, min_prob=0.05):
    candidates = {tgt: table[tgt].get(src_word, 0.0) for tgt in table}
    if not candidates:
        return None
    tgt, prob = max(candidates.items(), key=lambda kv: kv[1])
    return tgt if prob >= min_prob else None


def translate_statistical(text, table):
    words = re.findall(r"[a-záéíóúñ]+", text.lower())
    out = [best_translation(w, table) or f"*{w}" for w in words]
    return " ".join(out)


translation_stat = translate_statistical(FULL_TEXT_GL, ibm1_model.translation_table)
elapsed_stat = time.time() - start_time
emissions_stat_kg = tracker_stat.stop()

print(translation_stat)
print(f"\n({elapsed_stat:.4f}s, {emissions_stat_kg * 1000:.6f} g CO2eq)")
```

</details>

**Why this looks like word salad, and why that's the point:** repeated words like "stay
stay stay" or "with with with with" are not a bug in the decoding above --- they are a
real, well-documented property of translating word-by-word with no language model and no
reordering: several Galician words end up tied, or nearly tied, across multiple English
words in such a small training corpus, and independently picking the single most likely
English word for each one can pick the same word repeatedly. This is close to what the
earliest purely word-based statistical MT systems actually produced, and it is a real
part of why the field moved toward phrase-based systems and, eventually, systems with an
integrated language model choosing word order. Words marked with a leading `*` never
appeared on the Galician side of the training corpus at all --- genuinely unknown to this
tiny model, the same kind of gap Apertium showed with `*calado` earlier, just for a
different reason (missing training data here, rather than a missing dictionary entry
there).


## Step 12 — Approach 3: a dedicated neural model (2015-style architecture)

`Helsinki-NLP/opus-mt-gl-en` does exactly one job — Galician to English — using the
encoder-decoder-with-attention architecture. It is far smaller
than the general-purpose model below; the first run downloads it (roughly 300MB). This one
gets the **whole passage**.

In [ ]:
MT_MODEL_NAME = "Helsinki-NLP/opus-mt-gl-en"

mt_tokenizer = AutoTokenizer.from_pretrained(MT_MODEL_NAME)
mt_model = AutoModelForSeq2SeqLM.from_pretrained(MT_MODEL_NAME).to(model.device)
print(f"Loaded {MT_MODEL_NAME}")
print(f"Parameter count: {sum(p.numel() for p in mt_model.parameters()) / 1e6:.1f}M")


def translate_with_dedicated_model(text):
    inputs = mt_tokenizer(text, return_tensors="pt", truncation=True).to(mt_model.device)
    with torch.no_grad():
        output = mt_model.generate(**inputs, max_new_tokens=200)
    return mt_tokenizer.decode(output[0], skip_special_tokens=True)


tracker_neural = EmissionsTracker(
    project_name="green-mt-neural-dedicated",
    measure_power_secs=1,
    save_to_file=False,
    log_level="warning",
)
tracker_neural.start()
start_time = time.time()
translation_neural = translate_with_dedicated_model(FULL_TEXT_GL)
elapsed_neural = time.time() - start_time
emissions_neural_kg = tracker_neural.stop()

print(f"({elapsed_neural:.2f}s, {emissions_neural_kg * 1000:.4f} g CO2eq)\n")
print(translation_neural)

## Step 13 — Approach 4: the general-purpose LLM

The same model, same kind of prompt as Part 1 — just asked to translate instead of answer a
question.

In [ ]:
def translate_with_llm(text, source_lang="Galician", target_lang="English"):
    messages = [
        {
            "role": "system",
            "content": (
                f"You are a professional translator. Translate the user's {source_lang} "
                f"text into fluent {target_lang}. Output only the translation."
            ),
        },
        {"role": "user", "content": text},
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=200, do_sample=False)
    generated = output[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()


tracker_llm = EmissionsTracker(
    project_name="green-mt-general-llm",
    measure_power_secs=1,
    save_to_file=False,
    log_level="warning",
)
tracker_llm.start()
start_time = time.time()
translation_llm = translate_with_llm(FULL_TEXT_GL)
elapsed_llm = time.time() - start_time
emissions_llm_kg = tracker_llm.stop()

print(f"({elapsed_llm:.2f}s, {emissions_llm_kg * 1000:.4f} g CO2eq)\n")
print(translation_llm)

## Step 14 — Compare all four

Same underlying sentence, four very different systems and costs.

In [ ]:
mt_comparison = pd.DataFrame([
    {"era": "1. Rule-based (real)", "model": "Apertium (glg-eng)", "params": "n/a (rule-based)",
     "scope": "full passage", "duration_sec": round(elapsed_rule, 4),
     "emissions_g_co2eq": round(emissions_rule_kg * 1000, 6)},
    {"era": "2. Statistical", "model": "IBM Model 1, 24 phrase pairs", "params": "n/a",
     "scope": "full passage", "duration_sec": round(elapsed_stat, 4),
     "emissions_g_co2eq": round(emissions_stat_kg * 1000, 6)},
    {"era": "3. Neural, dedicated", "model": MT_MODEL_NAME,
     "params": f"{sum(p.numel() for p in mt_model.parameters()) / 1e6:.0f}M",
     "scope": "full passage", "duration_sec": round(elapsed_neural, 2),
     "emissions_g_co2eq": round(emissions_neural_kg * 1000, 4)},
    {"era": "4. General-purpose LLM", "model": MODEL_NAME,
     "params": f"{sum(p.numel() for p in model.parameters()) / 1e9:.2f}B",
     "scope": "full passage", "duration_sec": round(elapsed_llm, 2),
     "emissions_g_co2eq": round(emissions_llm_kg * 1000, 4)},
])

print(mt_comparison.to_string(index=False))

print("\nLook at the neural and LLM translations again. Does either introduce the aunt as a")
print("NEW character (something like 'I had an aunt who...'), or does it flatten her into")
print("'my aunt'? Fixing this requires restructuring the sentence — choosing better words or")
print("word order alone can't do it. That is the alignment-vs-translation gap from the")
print("lesson.")
print()
print("The neural and LLM models here are small on purpose (to run on a free Colab GPU), and")
print("often still flatten it too. That's a limit of THIS NOTEBOOK'S model size, worth")
print("testing directly: paste the Galician sentence into a large commercial model (ChatGPT,")
print("Claude, Gemini) and see whether it gets \"an aunt of mine\" or similar right — it")
print("often does. Scale and training data can close this specific gap; the rule-based and")
print("statistical approaches above have a structural ceiling that scale alone can't touch.")

### Seeing the cost spread

In [ ]:
import matplotlib.pyplot as plt

bar_colors = ["tab:orange" if era.startswith("1.") else "tab:blue" for era in mt_comparison["era"]]
cheapest = mt_comparison.loc[mt_comparison["emissions_g_co2eq"].idxmin()]
priciest = mt_comparison.loc[mt_comparison["emissions_g_co2eq"].idxmax()]
ratio = priciest["emissions_g_co2eq"] / max(cheapest["emissions_g_co2eq"], 1e-12)

plt.figure(figsize=(8, 4))
plt.bar(mt_comparison["era"], mt_comparison["emissions_g_co2eq"], color=bar_colors)
plt.yscale("log")
plt.ylabel("g CO2eq (log scale)")
plt.title(f"Priciest vs. cheapest translation approach: ~{ratio:,.0f}x apart")
plt.xticks(rotation=15, ha="right")
plt.tight_layout()
plt.show()

print(f"Cheapest: {cheapest['era']} ({cheapest['emissions_g_co2eq']:.6f} g CO2eq).")
print(f"Priciest: {priciest['era']} ({priciest['emissions_g_co2eq']:.4f} g CO2eq).")
print("The orange bar (rule-based) measures local network wait — Apertium's real")
print("server-side compute stays invisible to us here (see Step 10). Read it as a lower")
print("bound, a different kind of measurement from the other three bars.")

## Step 15 — Log this comparison

In [ ]:
mt_leaderboard = pd.DataFrame([
    {"task": "translation", "era": "rule-based", "model": "apertium-glg-eng",
     "duration_sec": elapsed_rule, "energy_kwh": get_tracker_energy_kwh(tracker_rule),
     "emissions_kg_co2eq": emissions_rule_kg},
    {"task": "translation", "era": "statistical", "model": "ibm-model-1",
     "duration_sec": elapsed_stat, "energy_kwh": get_tracker_energy_kwh(tracker_stat),
     "emissions_kg_co2eq": emissions_stat_kg},
    {"task": "translation", "era": "neural-dedicated", "model": MT_MODEL_NAME,
     "duration_sec": elapsed_neural, "energy_kwh": get_tracker_energy_kwh(tracker_neural),
     "emissions_kg_co2eq": emissions_neural_kg},
    {"task": "translation", "era": "general-llm", "model": MODEL_NAME,
     "duration_sec": elapsed_llm, "energy_kwh": get_tracker_energy_kwh(tracker_llm),
     "emissions_kg_co2eq": emissions_llm_kg},
])
mt_leaderboard.to_csv("mt_leaderboard.csv", index=False)
mt_leaderboard

In [ ]:
import os

if os.path.exists("mt_leaderboard.csv"):
    print("Saved mt_leaderboard.csv in this Colab session.")
    print("To download it: click the folder icon in the left sidebar, find")
    print("mt_leaderboard.csv, right-click it, and choose Download.")
else:
    print("mt_leaderboard.csv not found — re-run the cell above first.")

### Optional stretch goal — flip the direction, or change the text

If you have time:

- **Translate English into Galician instead**: Apertium also has a real English→Galician
  pair — pass `langpair="eng|glg"` to `translate_rule_based` in Step 10. For the neural
  model, set `MT_MODEL_NAME = "Helsinki-NLP/opus-mt-en-gl"` and re-run Step 12, and change
  `source_lang`/`target_lang` in Step 13's call to `translate_with_llm` to translate *into*
  Galician instead of out of it.
- **Use a longer or different text**: replace `FULL_TEXT_GL` in Step 9 with a longer
  passage (e.g. the rest of *Speechwriter*'s first chapter, if your instructor has shared
  it) or any other Galician text, and re-run Steps 10 and 13–16. Does the cost gap between
  the neural and general-purpose approaches grow, shrink, or stay about the same as the
  text gets longer? Why, given how attention cost scales with context length?

## Part 3 — Evaluating translations (and what evaluation itself costs)

So which translation was actually *good*? "Compare them yourself" doesn't scale past a
handful of sentences — real MT systems are evaluated automatically, against a reference
translation written by a human. We'll try three evaluation methods, from oldest and
cheapest to newest and most expensive, and measure the cost of **evaluating** too.

**Step 16 — The reference translation.** `REFERENCE_EN` below is one reasonable human translation, incorporating the "I had an aunt
who..." fix. Feel free to write your own instead — the point of this section
holds either way, and a metric is only as good as the reference it's compared against,
which is itself a real cost (a human had to write it).

In [ ]:
REFERENCE_EN = (
    "Sometimes it's better to stay quiet. I had an aunt who solved everything with sayings "
    "and proverbs; she used to say something like, 'You're a slave to what you say, and "
    "master of what you keep to yourself.' It always struck me as a wise saying, and maybe "
    "I should have listened to it on a few occasions — the rare ones that, even so, ended up "
    "sharply changing the course of my life."
)

candidates = {
    "rule-based": translation_rule,
    "statistical": translation_stat,
    "neural-dedicated": translation_neural,
    "general-llm": translation_llm,
}

for name, text in candidates.items():
    print(f"--- {name} ---")
    print(text)
    print()
print("--- reference ---")
print(REFERENCE_EN)

**Scope note:** `statistical` now translates the same `FULL_TEXT_GL` as the other three
systems, word-by-word with no reordering (see Step 11). Its BLEU/BERTScore/judge scores
below will still be low, but for an honest reason now: repeated and unknown (`*`-marked)
words, not a narrower scope than everyone else's. It is still worth scoring: the point is
to see the actual number a monotone, language-model-free decoder produces, not to assume
it.


## Step 17 — BLEU (2002): count overlapping n-grams

BLEU just counts how many word sequences (n-grams) a candidate translation shares with the
reference. No model, no GPU — pure counting, the same idea as n-gram counting.

### ✏️ Exercise 9 — BLEU for every candidate

Compute the BLEU score of each candidate translation against `REFERENCE_EN`.
`sacrebleu.corpus_bleu` takes a list of system outputs and a list of reference lists.

<details><summary><b>Hints</b> (click to open)</summary>

- The candidate goes in the first list, the reference in the nested second list.
- Swapping them gives a different number.

</details>

In [ ]:
!pip install -q sacrebleu
import sacrebleu

tracker_bleu = EmissionsTracker(
    project_name="green-eval-bleu",
    measure_power_secs=1,
    save_to_file=False,
    log_level="warning",
)
tracker_bleu.start()
start_time = time.time()

bleu_scores = ___  # TODO: {candidate name: BLEU score against REFERENCE_EN}

elapsed_bleu = time.time() - start_time
emissions_bleu_kg = tracker_bleu.stop()

for name, score in bleu_scores.items():
    print(f"{name}: BLEU = {score:.2f}")
print(f"\n({elapsed_bleu:.4f}s, {emissions_bleu_kg * 1000:.6f} g CO2eq for all candidates)")

In [ ]:
# Check for Exercise 9. Run this cell as is.
assert set(bleu_scores) == set(candidates)
assert all(0 <= s <= 100 for s in bleu_scores.values())
_n = "general-llm"
assert abs(bleu_scores[_n] - sacrebleu.corpus_bleu([candidates[_n]], [[REFERENCE_EN]]).score) < 1e-9, \
    "the candidate goes first, the reference second"
print("✓ Exercise 9 passed")

<details><summary><b>Solution to Exercise 9</b> (open it after you have tried)</summary>

```python
!pip install -q sacrebleu
import sacrebleu

tracker_bleu = EmissionsTracker(
    project_name="green-eval-bleu",
    measure_power_secs=1,
    save_to_file=False,
    log_level="warning",
)
tracker_bleu.start()
start_time = time.time()

bleu_scores = {
    name: sacrebleu.corpus_bleu([text], [[REFERENCE_EN]]).score
    for name, text in candidates.items()
}

elapsed_bleu = time.time() - start_time
emissions_bleu_kg = tracker_bleu.stop()

for name, score in bleu_scores.items():
    print(f"{name}: BLEU = {score:.2f}")
print(f"\n({elapsed_bleu:.4f}s, {emissions_bleu_kg * 1000:.6f} g CO2eq for all candidates)")
```

</details>

### ✏️ Exercise 10 — BLEU from its definition (advanced)

`sacrebleu` computed BLEU in one line. Here you compute it from its definition
(Papineni et al. 2002) and check that you get the same numbers.

1. **Clipped n-gram precision** p_n, for n = 1 … 4. Each candidate n-gram counts as correct
   at most as many times as it occurs in the reference:
   p_n = (correct n-grams) / (all n-grams in the candidate).
2. **Brevity penalty.** With candidate length c and reference length r, in tokens:
   BP = 1 if c ≥ r, and BP = exp(1 − r / c) if c < r.
3. **BLEU** = 100 · BP · exp( ¼ · (log p_1 + log p_2 + log p_3 + log p_4) ),
   that is, 100 · BP · the geometric mean of the four precisions.

If some p_n is 0, its log is −∞ and BLEU is 0. By default, sacrebleu replaces each zero
precision with a small number instead; the given function `smoothed_precisions` does this
the same way. Both texts are split into tokens with sacrebleu's own tokenizer, so that the
counts match.

<details><summary><b>Hints</b> (click to open)</summary>

- `tuple(tokens[i:i + n])` is the n-gram starting at position i; there are `len(tokens) - n + 1` of them.
- Clipping: `sum(min(count, ref[g]) for g, count in cand.items())`. A Counter returns 0 for a missing key.
- `math.exp(1 - r / c)` when `c < r`, else 1.
- The geometric mean of the precisions is `math.exp(sum(math.log(p) for p in precisions) / max_n)`.

</details>

In [ ]:
import math
from collections import Counter
from sacrebleu.tokenizers.tokenizer_13a import Tokenizer13a

tokenize_13a = Tokenizer13a()


def ngrams(tokens, n):
    # TODO: a Counter of the n-grams in tokens, each n-gram as a tuple of n tokens
    ___


def precision_counts(cand_tokens, ref_tokens, n):
    cand, ref = ngrams(cand_tokens, n), ngrams(ref_tokens, n)
    correct = ___  # TODO: matches, each n-gram clipped to its count in the reference
    total = ___  # TODO: the number of n-grams in the candidate
    return correct, total


def smoothed_precisions(correct, total):
    """Given: sacrebleu's default smoothing. Each zero count gets 1 / (2^k * total)."""
    precisions, factor = [], 1.0
    for c, t in zip(correct, total):
        if c == 0:
            factor *= 2
            precisions.append(1 / (factor * t))
        else:
            precisions.append(c / t)
    return precisions


def my_bleu(candidate, reference, max_n=4):
    cand_tokens = tokenize_13a(candidate).split()
    ref_tokens = tokenize_13a(reference).split()
    counts = [precision_counts(cand_tokens, ref_tokens, n) for n in range(1, max_n + 1)]
    correct = [c for c, _ in counts]
    total = [t for _, t in counts]
    if sum(correct) == 0:
        return 0.0
    c, r = len(cand_tokens), len(ref_tokens)
    bp = ___  # TODO: the brevity penalty
    precisions = smoothed_precisions(correct, total)
    # TODO: return 100 * BP * the geometric mean of the precisions
    ___


for name, text in candidates.items():
    print(f"{name:18s} mine = {my_bleu(text, REFERENCE_EN):6.2f}   sacrebleu = {bleu_scores[name]:6.2f}")

In [ ]:
# Check for Exercise 10. Run this cell as is.
assert ngrams(["a", "b", "a", "b"], 2) == Counter({("a", "b"): 2, ("b", "a"): 1})
assert precision_counts(["the", "the", "the"], ["the", "cat"], 1) == (1, 3), \
    "'the' occurs once in the reference, so it is correct at most once"
for _c, _r in [("the cat sat on a rug", "the cat sat on the mat"), ("on the mat the cat sat", "the cat sat on the mat")]:
    assert abs(my_bleu(_c, _r) - sacrebleu.corpus_bleu([_c], [[_r]]).score) < 1e-6, (_c, my_bleu(_c, _r))
for _name, _text in candidates.items():
    assert abs(my_bleu(_text, REFERENCE_EN) - bleu_scores[_name]) < 1e-6, _name
print("✓ Exercise 10 passed")

<details><summary><b>Solution to Exercise 10</b> (open it after you have tried)</summary>

```python
import math
from collections import Counter
from sacrebleu.tokenizers.tokenizer_13a import Tokenizer13a

tokenize_13a = Tokenizer13a()


def ngrams(tokens, n):
    return Counter(tuple(tokens[i:i + n]) for i in range(len(tokens) - n + 1))


def precision_counts(cand_tokens, ref_tokens, n):
    cand, ref = ngrams(cand_tokens, n), ngrams(ref_tokens, n)
    correct = sum(min(count, ref[g]) for g, count in cand.items())
    total = sum(cand.values())
    return correct, total


def smoothed_precisions(correct, total):
    """Given: sacrebleu's default smoothing. Each zero count gets 1 / (2^k * total)."""
    precisions, factor = [], 1.0
    for c, t in zip(correct, total):
        if c == 0:
            factor *= 2
            precisions.append(1 / (factor * t))
        else:
            precisions.append(c / t)
    return precisions


def my_bleu(candidate, reference, max_n=4):
    cand_tokens = tokenize_13a(candidate).split()
    ref_tokens = tokenize_13a(reference).split()
    counts = [precision_counts(cand_tokens, ref_tokens, n) for n in range(1, max_n + 1)]
    correct = [c for c, _ in counts]
    total = [t for _, t in counts]
    if sum(correct) == 0:
        return 0.0
    c, r = len(cand_tokens), len(ref_tokens)
    bp = 1.0 if c >= r else math.exp(1 - r / c)
    precisions = smoothed_precisions(correct, total)
    return 100 * bp * math.exp(sum(math.log(p) for p in precisions) / max_n)


for name, text in candidates.items():
    print(f"{name:18s} mine = {my_bleu(text, REFERENCE_EN):6.2f}   sacrebleu = {bleu_scores[name]:6.2f}")
```

</details>

### Add the BLEU score to `mt_leaderboard.csv`

Step 15 saved `mt_leaderboard.csv` before this BLEU score existed --- Part 3 runs after
Part 2. This adds it onto the matching rows now, and labels the `general-llm` row
`baseline-fp16`, so Session 2 has something concrete to compare its quantized run
against on both cost and quality.


In [ ]:
mt_leaderboard = pd.read_csv("mt_leaderboard.csv")
mt_leaderboard["bleu_score"] = mt_leaderboard["era"].map(bleu_scores)
mt_leaderboard.loc[mt_leaderboard["era"] == "general-llm", "config"] = "baseline-fp16"
mt_leaderboard.to_csv("mt_leaderboard.csv", index=False)
mt_leaderboard


## Step 18 — BERTScore (2020): a learned, neural metric

BERTScore compares the candidate and the reference using contextual embeddings from a
pretrained language model, matching words by cosine similarity instead of counting exact
overlaps — so it can reward a *correct paraphrase* that BLEU would penalize for not using
the reference's exact words. One real difference from a metric like COMET: BERTScore only
looks at candidate vs. reference, never the source sentence — a genuine methodological
limitation worth remembering when you read the score below, since a fluent-sounding
translation could still be unfaithful to the Galician original and BERTScore alone
wouldn't catch it. We use a smaller model here (`distilbert-base-uncased`) for speed; the
biggest model isn't needed to see the point.

**Known limitation, worth noticing:** this notebook used to use COMET here. Checked
directly against `unbabel-comet`'s own published dependency spec: it pins `numpy<2.0`,
`protobuf<5.0`, and `pytorch-lightning<3.0` — versions old enough now to reliably conflict
with Colab's current base image and crash on import. BERTScore's dependencies are
essentially unpinned, so this should actually run. We keep the same try/except pattern
anyway, since any live network install can still flake mid-class.

In [ ]:
!pip install -q bert-score

bertscore_available = False
bertscore_scores = {}
elapsed_bertscore = 0.0
emissions_bertscore_kg = 0.0

try:
    from bert_score import score as bertscore

    tracker_bertscore = EmissionsTracker(
        project_name="green-eval-bertscore",
        measure_power_secs=1,
        save_to_file=False,
        log_level="warning",
    )
    tracker_bertscore.start()
    start_time = time.time()

    cand_list = list(candidates.values())
    ref_list = [REFERENCE_EN] * len(cand_list)
    _, _, f1 = bertscore(
        cand_list, ref_list, model_type="distilbert-base-uncased", verbose=False
    )
    bertscore_scores = dict(zip(candidates.keys(), f1.tolist()))

    elapsed_bertscore = time.time() - start_time
    emissions_bertscore_kg = tracker_bertscore.stop()
    bertscore_available = True

    for name, s in bertscore_scores.items():
        print(f"{name}: BERTScore F1 = {s:.4f}")
    print(f"\n({elapsed_bertscore:.2f}s, {emissions_bertscore_kg * 1000:.4f} g CO2eq for all candidates)")
except Exception as e:
    print(f"BERTScore failed to load/run in this environment ({type(e).__name__}: {e}).")
    print("Continuing without a BERTScore score — Step 20 will compare BLEU and the LLM")
    print("judge only for this run.")

## Step 19 — LLM-as-a-judge: prompt a model to grade the translation

The newest approach: ask a general-purpose LLM to read the source, the candidate, and the
reference, and output a quality score itself. We reuse the same model from Parts 1 and 2 —
no new download.

In [ ]:
def llm_judge_score(source, candidate, reference):
    messages = [
        {
            "role": "system",
            "content": (
                "You are an expert translation evaluator. Given a source text, a candidate "
                "translation, and a reference translation, rate the candidate's quality "
                "from 1 (very poor) to 10 (excellent), considering both accuracy and "
                "fluency. Output ONLY the number."
            ),
        },
        {
            "role": "user",
            "content": (
                f"Source (Galician): {source}\n\n"
                f"Reference translation: {reference}\n\n"
                f"Candidate translation: {candidate}\n\n"
                "Score (1-10):"
            ),
        },
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=10, do_sample=False)
    generated = output[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()


tracker_judge = EmissionsTracker(
    project_name="green-eval-llm-judge",
    measure_power_secs=1,
    save_to_file=False,
    log_level="warning",
)
tracker_judge.start()
start_time = time.time()

judge_scores = {
    name: llm_judge_score(FULL_TEXT_GL, text, REFERENCE_EN)
    for name, text in candidates.items()
}

elapsed_judge = time.time() - start_time
emissions_judge_kg = tracker_judge.stop()

for name, score in judge_scores.items():
    print(f"{name}: LLM-judge score = {score}")
print(f"\n({elapsed_judge:.2f}s, {emissions_judge_kg * 1000:.4f} g CO2eq for all candidates)")

### A second, independent judge: ChatGPT, by hand

The score above came from the same Qwen model already loaded in this notebook grading
its own family's output --- a real, known weak spot of LLM-as-a-judge: a model can be
biased toward whatever its own family tends to produce. To check that, get a second
opinion from a genuinely different, independent model: ChatGPT (or Claude, Gemini,
whichever you have access to), run by hand, outside this notebook.

The cell below prints one ready-to-paste prompt per candidate --- the exact same prompt
Qwen was just given. Paste each one into your chat tool of choice, one at a time, then
fill the numbers it returns into `CHATGPT_JUDGE_SCORES` in the cell after that.


In [ ]:
def build_judge_prompt(source, candidate, reference):
    return (
        "You are an expert translation evaluator. Given a source text, a candidate "
        "translation, and a reference translation, rate the candidate's quality from 1 "
        "(very poor) to 10 (excellent), considering both accuracy and fluency. Output "
        "ONLY the number.\n\n"
        f"Source (Galician): {source}\n\n"
        f"Reference translation: {reference}\n\n"
        f"Candidate translation: {candidate}\n\n"
        "Score (1-10):"
    )


for name, text in candidates.items():
    print(f"===== prompt for: {name} =====")
    print(build_judge_prompt(FULL_TEXT_GL, text, REFERENCE_EN))
    print()


Replace each `None` below with the number your chat tool returned for that candidate.


In [ ]:
CHATGPT_JUDGE_SCORES = {
    "rule-based": None,
    "statistical": None,
    "neural-dedicated": None,
    "general-llm": None,
}

chatgpt_judge_scores = CHATGPT_JUDGE_SCORES
assert all(v is not None for v in chatgpt_judge_scores.values()), (
    "Fill in every score in CHATGPT_JUDGE_SCORES above before continuing --- some are still None."
)
for name, score in chatgpt_judge_scores.items():
    print(f"{name}: LLM-judge score (ChatGPT, by hand) = {score}")

elapsed_chatgpt_judge = None
emissions_chatgpt_judge_kg = None


## Step 20 — Compare the four evaluation methods

Same four candidate translations, four ways of scoring them — three with a measurable
local cost, one (the ChatGPT judge) that has no local cost by design, since the actual
grading ran outside this notebook.

### What we're actually scoring

The numbers below are only meaningful next to the text they're scoring — here it is
again, right next to the comparison.

In [ ]:
for name, text in candidates.items():
    print(f"{name}:")
    print(f"  {text}")
    print()
print("reference:")
print(f"  {REFERENCE_EN}")

In [ ]:
eval_rows = [
    {"metric": "BLEU (2002)", "needs_gpu": False,
     "duration_sec": round(elapsed_bleu, 4),
     "emissions_g_co2eq": round(emissions_bleu_kg * 1000, 6),
     **{f"{k}_score": round(v, 2) for k, v in bleu_scores.items()}},
]
if bertscore_available:
    eval_rows.append({"metric": "BERTScore (2020)", "needs_gpu": True,
     "duration_sec": round(elapsed_bertscore, 2),
     "emissions_g_co2eq": round(emissions_bertscore_kg * 1000, 4),
     **{f"{k}_score": round(v, 4) for k, v in bertscore_scores.items()}})
eval_rows.append({"metric": "LLM-as-judge (Qwen, today)", "needs_gpu": True,
     "duration_sec": round(elapsed_judge, 2),
     "emissions_g_co2eq": round(emissions_judge_kg * 1000, 4),
     **{f"{k}_score": v for k, v in judge_scores.items()}})
eval_rows.append({"metric": "LLM-as-judge (ChatGPT, by hand)", "needs_gpu": False,
     "duration_sec": elapsed_chatgpt_judge,
     "emissions_g_co2eq": emissions_chatgpt_judge_kg,
     **{f"{k}_score": v for k, v in chatgpt_judge_scores.items()}})

eval_comparison = pd.DataFrame(eval_rows)

print(eval_comparison.to_string(index=False))
if not bertscore_available:
    print("\n(BERTScore is missing from this run — it failed to load in Step 18. This is")
    print("BLEU vs. the LLM judge only.)")
print("\nDo the metrics you have agree on which candidate is better? If not, which would")
print("you trust — and would your answer change if you were scoring 10,000 translations")
print("instead of 2, and had to pay for the energy each method uses?")

### One bar chart per metric

Same idea as the table above, but each metric gets its own panel and its own scale ---
BLEU is only ever compared to BLEU, BERTScore only to BERTScore, and the LLM judge only to
itself, across the four systems. The three metrics are not on comparable scales (BLEU is
roughly 0-100, BERTScore F1 is 0-1, the judge is 1-10), so they are never plotted together
on one axis.


### ✏️ Exercise 11 — Read a number out of a judge's reply

The LLM judge may answer `"8"`, `"Score: 7.5"` or `"7/10"`. Complete `parse_score`
so it returns the first number in the reply as a float, or `None` if there is no number.

<details><summary><b>Hints</b> (click to open)</summary>

- `re.search(pattern, text)` returns the first match, or `None`.
- `r"\d+"` matches one or more digits; `(\.\d+)?` adds an optional decimal part.

</details>

In [ ]:
import re


def parse_score(value):
    try:
        return float(value)
    except (TypeError, ValueError):
        # TODO: the first number in str(value), as a float, or None
        ___


score_sets = [("BLEU", bleu_scores)]
if bertscore_available:
    score_sets.append(("BERTScore", bertscore_scores))
score_sets.append(("LLM-judge (Qwen)", {k: parse_score(v) for k, v in judge_scores.items()}))
score_sets.append(("LLM-judge (ChatGPT)", {k: parse_score(v) for k, v in chatgpt_judge_scores.items()}))

fig, axes = plt.subplots(1, len(score_sets), figsize=(4.5 * len(score_sets), 4))
if len(score_sets) == 1:
    axes = [axes]

for ax, (metric_name, scores) in zip(axes, score_sets):
    names = list(scores.keys())
    values = [scores[n] for n in names]
    ax.bar(names, values)
    ax.set_title(metric_name)
    ax.tick_params(axis="x", rotation=30)

plt.tight_layout()
plt.show()


In [ ]:
# Check for Exercise 11. Run this cell as is.
assert parse_score("8") == 8.0
assert parse_score("Score: 7.5") == 7.5
assert parse_score("7/10") == 7.0
assert parse_score("no idea") is None
assert parse_score(None) is None
print("✓ Exercise 11 passed")

<details><summary><b>Solution to Exercise 11</b> (open it after you have tried)</summary>

```python
import re


def parse_score(value):
    try:
        return float(value)
    except (TypeError, ValueError):
        match = re.search(r"-?\d+(\.\d+)?", str(value))
        return float(match.group()) if match else None


score_sets = [("BLEU", bleu_scores)]
if bertscore_available:
    score_sets.append(("BERTScore", bertscore_scores))
score_sets.append(("LLM-judge (Qwen)", {k: parse_score(v) for k, v in judge_scores.items()}))
score_sets.append(("LLM-judge (ChatGPT)", {k: parse_score(v) for k, v in chatgpt_judge_scores.items()}))

fig, axes = plt.subplots(1, len(score_sets), figsize=(4.5 * len(score_sets), 4))
if len(score_sets) == 1:
    axes = [axes]

for ax, (metric_name, scores) in zip(axes, score_sets):
    names = list(scores.keys())
    values = [scores[n] for n in names]
    ax.bar(names, values)
    ax.set_title(metric_name)
    ax.tick_params(axis="x", rotation=30)

plt.tight_layout()
plt.show()
```

</details>

In [ ]:
cheapest_eval = eval_comparison.loc[eval_comparison["emissions_g_co2eq"].idxmin()]
priciest_eval = eval_comparison.loc[eval_comparison["emissions_g_co2eq"].idxmax()]
eval_ratio = priciest_eval["emissions_g_co2eq"] / max(cheapest_eval["emissions_g_co2eq"], 1e-12)

plt.figure(figsize=(7, 4))
plt.bar(eval_comparison["metric"], eval_comparison["emissions_g_co2eq"])
plt.yscale("log")
plt.ylabel("g CO2eq (log scale)")
plt.title(f"Grading cost: ~{eval_ratio:,.0f}x apart, cheapest to priciest")
plt.xticks(rotation=15, ha="right")
plt.tight_layout()
plt.show()
print("(The ChatGPT judge has no bar above --- its cost was never local to this notebook")
print("to measure, not zero.)")

print(f"Cheapest to grade with: {cheapest_eval['metric']} ({cheapest_eval['emissions_g_co2eq']:.6f} g CO2eq).")
print(f"Priciest to grade with: {priciest_eval['metric']} ({priciest_eval['emissions_g_co2eq']:.4f} g CO2eq).")

## Step 21 — Log the evaluation comparison

In [ ]:
# One row per (metric, candidate): duration/energy/emissions were measured once per
# metric, covering every candidate scored in that call together, not one candidate alone
# --- so the cost columns repeat across candidate-rows that share a metric, and only
# "score" varies row to row. This is what lets a later session compare an actual quality
# number against this baseline, not just what grading cost.
leaderboard_rows = []
for name, score in bleu_scores.items():
    leaderboard_rows.append({
        "metric": "bleu", "candidate": name, "score": round(score, 2),
        "duration_sec": elapsed_bleu,
        "energy_kwh": get_tracker_energy_kwh(tracker_bleu), "emissions_kg_co2eq": emissions_bleu_kg,
    })
if bertscore_available:
    for name, score in bertscore_scores.items():
        leaderboard_rows.append({
            "metric": "bertscore", "candidate": name, "score": round(score, 4),
            "duration_sec": elapsed_bertscore,
            "energy_kwh": get_tracker_energy_kwh(tracker_bertscore), "emissions_kg_co2eq": emissions_bertscore_kg,
        })
for name, score in judge_scores.items():
    leaderboard_rows.append({
        "metric": "llm_judge_qwen", "candidate": name, "score": score,
        "duration_sec": elapsed_judge,
        "energy_kwh": get_tracker_energy_kwh(tracker_judge), "emissions_kg_co2eq": emissions_judge_kg,
    })
for name, score in chatgpt_judge_scores.items():
    leaderboard_rows.append({
        "metric": "llm_judge_chatgpt", "candidate": name, "score": score,
        "duration_sec": elapsed_chatgpt_judge,
        "energy_kwh": None, "emissions_kg_co2eq": emissions_chatgpt_judge_kg,
    })

eval_leaderboard = pd.DataFrame(leaderboard_rows)
eval_leaderboard.to_csv("mt_eval_leaderboard.csv", index=False)
eval_leaderboard

In [ ]:
import os

if os.path.exists("mt_eval_leaderboard.csv"):
    print("Saved mt_eval_leaderboard.csv in this Colab session.")
    print("To download it: click the folder icon in the left sidebar, find")
    print("mt_eval_leaderboard.csv, right-click it, and choose Download.")
else:
    print("mt_eval_leaderboard.csv not found — re-run the cell above first.")

## Wrap-up: everything we measured today

Same units (g CO2eq), every technique from this notebook, on one chart.

In [ ]:
summary_rows = [
    {"item": "Part 1: QA baseline (8 questions)", "g_co2eq": emissions_kg * 1000},
    {"item": "Part 2: rule-based (Apertium)", "g_co2eq": emissions_rule_kg * 1000},
    {"item": "Part 2: statistical (toy)", "g_co2eq": emissions_stat_kg * 1000},
    {"item": "Part 2: neural, dedicated", "g_co2eq": emissions_neural_kg * 1000},
    {"item": "Part 2: general-purpose LLM", "g_co2eq": emissions_llm_kg * 1000},
    {"item": "Part 3: BLEU", "g_co2eq": emissions_bleu_kg * 1000},
]
if bertscore_available:
    summary_rows.append({"item": "Part 3: BERTScore", "g_co2eq": emissions_bertscore_kg * 1000})
summary_rows.append({"item": "Part 3: LLM-as-judge", "g_co2eq": emissions_judge_kg * 1000})

summary = pd.DataFrame(summary_rows)
overall_cheapest = summary.loc[summary["g_co2eq"].idxmin()]
overall_priciest = summary.loc[summary["g_co2eq"].idxmax()]
overall_ratio = overall_priciest["g_co2eq"] / max(overall_cheapest["g_co2eq"], 1e-12)

plt.figure(figsize=(9, 5))
plt.barh(summary["item"], summary["g_co2eq"])
plt.xscale("log")
plt.xlabel("g CO2eq (log scale)")
plt.title(f"Cheapest to priciest technique today: ~{overall_ratio:,.0f}x apart")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

print(summary.to_string(index=False))
print(f"\nCheapest: {overall_cheapest['item']}. Priciest: {overall_priciest['item']}.")

**Key takeaways**

- Cost spans several orders of magnitude in this notebook, driven by *how* a system is
  built — fixed rules, counting word pairs, a small trained network, or a giant
  general-purpose model each cost fundamentally different amounts to run, independent of
  output quality.
- Model size and cost move independently of each other — now with numbers you measured
  yourself.
- Evaluation has its own cost curve, separate from translation's: BLEU is nearly free, a
  learned neural metric costs real energy, and an LLM-as-judge sits somewhere in between
  depending on the model. Grading carries a real cost too, on top of translating.
- Even "free" or "off-the-shelf" tools carry real friction: Apertium's real compute stays
  invisible to a local CodeCarbon tracker, COMET's pinned dependencies reliably break on
  current Colab images, and CodeCarbon itself falls back to rougher estimates when it
  can't read your GPU or your location. Measuring your own numbers, and understanding
  where those numbers come from, is the actual skill this notebook practices.
- Handing the model the documents is what makes the quiz answerable: the closed-book
  run shows how little of it the model knows on its own, and how confidently it can
  be wrong.
- `leaderboard.csv` is the empirical baseline later comparisons build on — the chart back
  in Step 7 shows where new rows slot in. Same questions, same scoring, one change at a
  time, using the numbers you personally measured today.

## Step 22 — Share-out (discussion)

Before we wrap up, compare notes with your neighbor.

**On the QA baseline (Part 1):**

- What energy and emissions numbers did you get? How close were they to each other's, given
  you likely used the same model on similar Colab hardware?
- What was your accuracy? If it wasn't 100%, look at the wrong answers — what went wrong?
- How many questions did the model answer with no documents? When it was wrong, did it
  say so, or did it give a confident answer anyway?
- Did CodeCarbon report any warnings about not recognizing your hardware? What does that
  tell you about trusting a single carbon number at face value?

**On the translation comparison (Part 2):**

- Look at the cost table from Step 14. How many orders of magnitude separate the cheapest
  and most expensive approach?
- Neither the real rule-based system nor the toy statistical system could ever produce
  "I had an aunt who..." — why not, structurally? What would it actually take to fix
  that, beyond a bigger dictionary or a better language model?
- Apertium is a real, currently maintained system — did that change how seriously you
  took its limitations, compared to the rule-based approach you might have pictured
  before running this?
- Was the dedicated neural model's translation actually worse than the general-purpose
  LLM's, or just different?
- This notebook's general-purpose LLM is small (1.5B parameters), to keep things running
  on a free Colab GPU. Paste the Galician sentence into a large commercial model (ChatGPT,
  Claude, Gemini) and compare: does it get "an aunt of mine" or similar right, where our
  small local model didn't? What does that tell you about whether this was a limit of
  neural/LLM translation in general, or just a limit of *this notebook's* model size?

**On the evaluation comparison (Part 3):**

- Did BLEU, BERTScore, and the LLM judge agree on which translation was better?
- Which one would you trust for a single sentence? For 10,000 sentences, where the cost of
  scoring itself starts to matter?

## Step 23 — Code literacy check

Before you're done, pick at least two or three code cells above that you actually ran and
read today — across any of Parts 1-3 — and add `#` comments to them explaining, in your
own words, what each meaningful line or block does and *why* it's there, in language a
classmate who hasn't seen the code could follow.

Do this part on your own, after class:

1. Paste your commented code into an LLM chat (ChatGPT, Claude, Gemini — whichever you
   have access to).
2. Ask it something like: *"Here is code I commented myself, explaining what I think it
   does. Based only on my comments, how well do you think I understood this code? What
   did I get right, what did I miss, and what's still fuzzy?"*
3. Read what it says. Where it disagrees with your comment, go check the code again —
   who's right?

This is a self-check on your own understanding: a way to catch, in your own words, where
your mental model of the code breaks down against a second reader.